# Propuesta de Temas y Hoja de Ruta (Roadmap) - Proyecto Integrador

**Curso:** Ingeniería de Datos  
**Maestría:** Ciencia de Datos e Inteligencia Artificial (DSAI - UTEC)  
**Docente:** Mg. Angel Tintaya Monroy  
**Objetivo:** Desarrollar el avance del proyecto integrador (Pipeline de Ingesta en Apache Airflow hacia Azure Data Lake Storage) y diseñar la hoja de ruta para la entrega del Sábado 06 de Junio.


## 1. Contexto General del Proyecto (Avance)

El proyecto consiste en diseñar y automatizar un pipeline de datos que extraiga archivos desde un servidor local simulado (una carpeta en local) y los cargue en un contenedor `Raw` (datos crudos) en la nube (**Azure Data Lake Storage - ADLS**) utilizando **Apache Airflow** como orquestador.

Este avance evalúa principalmente:
1. **Planteamiento del problema y contexto (25%):** Claridad al explicar el problema de negocio y su relevancia.
2. **Diseño del pipeline y arquitectura conceptual (20%):** Coherencia y justificación del flujo de datos y sus componentes.
3. **Pipeline funcional (20%):** Se evalúa si el DAG se ejecuta con éxito.
4. **Presentación clara y estructurada (15%):** Claridad en la presentación, enfoque estructurado (**SIN CÓDIGO** en las diapositivas).
5. **Uso de Git y documentación básica (20%):** Repositorio en GitHub con Git Flow y Pull Request.


## 2. Temas Potenciales de Negocio (Propuestas)

Para cumplir de manera sobresaliente con el criterio "Planteamiento del problema y contexto", debemos enmarcar el flujo en una necesidad de negocio real. A continuación, se proponen 4 temas potenciales:

### 💳 Tema A: Ingestión de Transacciones de Ventas (E-Commerce / Retail)
* **Caso de Uso:** Una cadena de retail genera archivos CSV diarios de transacciones en sus servidores POS locales. El equipo de analítica requiere consolidar estos datos en Azure Data Lake en la capa `Raw` para luego limpiarlos, procesarlos y actualizar tableros de ventas y control de inventario.
* **Estructura del Archivo (CSV):** `id_transaccion`, `id_cliente`, `producto`, `cantidad`, `monto_total`, `tienda`, `fecha_hora`.
* **Justificación de Automatización:** La carga manual de transacciones es propensa a errores, tiene retrasos y retrasa el análisis del stock disponible. Automatizar esto diariamente permite alertas rápidas de falta de stock y cálculo de KPIs de venta nocturnos de manera consistente.

### 📞 Tema B: Telemetría de Sensores de Maquinaria (Industrial IoT)
* **Caso de Uso:** Una planta industrial tiene sensores en sus máquinas de ensamblaje que vuelcan archivos JSON con telemetría cada hora en un servidor local. Se deben migrar a la nube para ejecutar análisis de anomalías e implementar mantenimiento predictivo.
* **Estructura del Archivo (JSON):** `sensor_id`, `temperatura`, `vibracion`, `presion`, `codigo_error`, `timestamp`.
* **Justificación de Automatización:** El volumen de datos IoT es masivo y constante. La automatización asegura la disponibilidad inmediata de las lecturas para prevenir fallas catastróficas en la producción y paradas no planificadas.

### 💻 Tema C: Clickstream de Navegación Web (Análisis de Comportamiento)
* **Caso de Uso:** Una plataforma de e-commerce almacena los logs de las interacciones de los usuarios en archivos semiestructurados (NDJSON) en el servidor web local. Subirlos a la nube permite procesar el comportamiento de navegación para personalizar ofertas y optimizar la tasa de conversión.
* **Estructura del Archivo (JSON):** `session_id`, `user_id`, `url_visitada`, `accion_realizada`, `dispositivo`, `timestamp`.
* **Justificación de Automatización:** Analizar el comportamiento del usuario requiere procesar flujos continuos de datos. Automatizar la carga a la nube permite que los modelos de recomendación y las métricas de conversión se actualicen de manera periódica sin intervención humana.

### 📊 Tema D: Datos de Precios de Criptomonedas / Acciones (Fintech)
* **Caso de Uso:** Una fintech descarga información histórica de precios de mercado cada hora y la guarda en CSVs locales. Se requiere enviar estos datos crudos a Azure para habilitar modelos de inversión cuantitativos.
* **Estructura del Archivo (CSV):** `ticker`, `precio_apertura`, `precio_maximo`, `precio_minimo`, `precio_cierre`, `volumen`, `timestamp`.
* **Justificación de Automatización:** En finanzas, el tiempo y la consistencia de los datos históricos son críticos. Automatizar la ingesta evita la pérdida de registros históricos y acelera el reentrenamiento de algoritmos de inversión.


## 3. Arquitectura Conceptual del Pipeline

El flujo conceptual de datos automatizado con Apache Airflow es el siguiente:

```
  +------------------------+      +---------------------------------+      +------------------------+
  |     Servidor Local     |      |          Apache Airflow         |      |       Azure Cloud      |
  |  (Simulador Local Dir) |      |           (Orquestador)         |      |     (Data Lake Gen2)   |
  |                        |      |                                 |      |                        |
  |  +------------------+  |      |  +---------------------------+  |      |  +------------------+  |
  |  | Carpeta de Datos |  | ---> |  |  FileSensor               |  |      |  | Contenedor: raw  |  |
  |  |  (datos/input/)  |  |      |  |  (Detecta archivos nuevos)|  |      |  +------------------+  |
  |  +------------------+  |      |  +---------------------------+  |      |            ^           |
  +------------------------+      |                |                |      |            |           |
                                  |                v                |      |            |           |
                                  |  +---------------------------+  |      |            |           |
                                  |  | LocalToWasbOperator       |  |------+------------+           |
                                  |  | (Sube archivos a Azure)   |  | (Carga segura en la nube)  
                                  |  +---------------------------+  |                                
                                  |                |                |                                
                                  |                v                |                                
                                  |  +---------------------------+  |                                
                                  |  | PythonOperator (Archive)  |  |                                
                                  |  | (Mueve local a /archive)  |  |                                
                                  |  +---------------------------+  |                                
                                  +---------------------------------+                                
```

### Componentes Principales:
1. **Servidor Local (Origen):** Directorio local simulado donde se depositan los datos de negocio.
2. **Apache Airflow (Orquestador):** Controla el flujo completo: detecta la presencia de archivos, inicia la transferencia a la nube y realiza tareas de ordenamiento (archivado de archivos locales procesados) para evitar duplicaciones.
3. **Azure Data Lake Storage (Destino):** Cuenta de almacenamiento en Azure (`stdemdsai`) con un contenedor llamado `raw` donde se guardan los datos limpios en su formato original.


## 4. Hoja de Ruta (Roadmap) del Proyecto

Cronograma de actividades sugerido para llegar preparados a la exposición del **Sábado 06 de Junio, 14:30 pm** (Duración máxima: 10 minutos de exposición).

| Fase / Fecha sugerida | Actividad / Hito | Responsable Sugerido | Entregable / Evidencia |
| :--- | :--- | :--- | :--- |
| **Fase 1: Configuración de Datos** <br> *(Miercoles 03 / Jueves 04)* | 1. Definición grupal de la temática (se recomienda **Tema A: Ventas Retail**). <br> 2. Implementar el generador de CSVs ficticios para simular el origen. <br> 3. Crear carpetas locales `datos_simulados/input` y `datos_simulados/archive`. | Integrante 1 | Script generador ejecutado y datos simulados listos localmente. |
| **Fase 2: Conectividad y Cloud** <br> *(Jueves 04)* | 1. Extraer la cadena de conexión de `azure_connection.txt`. <br> 2. Escribir un script de Python de prueba con la librería `azure-storage-blob` para verificar accesos. <br> 3. Asegurar que existe el contenedor `raw` en la cuenta `stdemdsai`. | Integrante 2 | Script de prueba con conexión exitosa a Azure Storage. |
| **Fase 3: Desarrollo del DAG** <br> *(Viernes 05)* | 1. Crear el DAG en el repositorio. <br> 2. Configurar la conexión `wasb_default` en Airflow UI usando la cadena de conexión. <br> 3. Definir y encadenar las tareas: `FileSensor` (para escuchar el CSV) -> `LocalFilesystemToWasbOperator` (para subirlo a Azure) -> `PythonOperator` (para mover el archivo procesado local a `archive/`). <br> 4. Validar la ejecución exitosa del DAG. | Integrante 3 e Integrante 4 | DAG funcional en Airflow (sin errores en el log). |
| **Fase 4: Git y Colaboración** <br> *(Viernes 05 / Sábado 06 mañana)* | 1. Crear la rama `feature/` de desarrollo en Git. <br> 2. Seguir flujo de Git Flow para subir el código al repositorio `dsai-de-airflow`. <br> 3. Crear el Pull Request (PR) y redactar el README.md básico. <br> 4. Capturar pantalla del PR para la presentación. | Integrante 5 | Enlace al PR de Git y captura de pantalla lista. |
| **Fase 5: Diapositivas y Ensayo** <br> *(Sábado 06)* | 1. Completar la plantilla `DE_Proyecto_Avance.pptx` (máximo 15 diapositivas). <br> 2. Asegurarse de que el enfoque de la presentación sea conceptual (negocio, arquitectura, flujos) y **no técnico** (sin código). <br> 3. Insertar la arquitectura conceptual y las capturas de pantalla del DAG ejecutado y del PR de Git. <br> 4. Ensayar la exposición grupal de 10 minutos. | Todo el grupo | Diapositivas listas y ensayo grupal completado. |


## 5. Herramientas Prácticas para el Desarrollo

A continuación, se proporcionan los bloques de código listos para su uso y adaptación por parte del equipo.


In [ ]:
# BLOQUE 1: GENERADOR DE DATOS FICTICIOS (CASO VENTAS RETAIL)
# Este script genera un archivo CSV simulando una carga diaria de ventas.
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

# Crear directorios de simulación local
INPUT_DIR = "datos_simulados/input"
ARCHIVE_DIR = "datos_simulados/archive"
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

# Parámetros de simulación
productos = ["Laptop", "Mouse Optico", "Teclado Mecanico", "Monitor 24'", "Audifonos", "Disco SSD 1TB"]
tiendas = ["Barranco", "Miraflores", "San Isidro", "Santiago de Surco", "La Molina"]
num_filas = 50

# Generación aleatoria
data = {
    "id_transaccion": [f"TRX-{i:05d}" for i in range(1, num_filas + 1)],
    "id_cliente": [f"CLI-{random.randint(10000, 99999)}" for _ in range(num_filas)],
    "producto": [random.choice(productos) for _ in range(num_filas)],
    "cantidad": [random.randint(1, 4) for _ in range(num_filas)],
    "monto_total": [round(random.uniform(15.0, 1200.0), 2) for _ in range(num_filas)],
    "tienda": [random.choice(tiendas) for _ in range(num_filas)],
    "fecha_hora": [(datetime.now() - timedelta(minutes=random.randint(10, 1440))).strftime('%Y-%m-%d %H:%M:%S') for _ in range(num_filas)]
}

df = pd.DataFrame(data)
filename = f"ventas_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
filepath = os.path.join(INPUT_DIR, filename)
df.to_csv(filepath, index=False)

print(f"[EXITO] Archivo de ventas simulado creado en: {filepath}")
print(df.head(5))


In [ ]:
# BLOQUE 2: SCRIPT DE PRUEBA DE CONEXIÓN A AZURE DATA LAKE
# Nota: Para ejecutar esto en local, primero debes instalar: pip install azure-storage-blob
import json
import os
from azure.storage.blob import BlobServiceClient

connection_file = "azure_connection.txt"

if not os.path.exists(connection_file):
    print(f"[ERROR] No se encontro el archivo '{connection_file}' en el directorio actual.")
else:
    try:
        # Leer la cadena de conexión
        with open(connection_file, 'r') as f:
            conn_data = json.load(f)
        connection_string = conn_data.get("connection_string")
        
        print("Estableciendo conexion con Azure Blob/Data Lake Storage...")
        blob_service_client = BlobServiceClient.from_connection_string(connection_string)
        
        # Crear un archivo de prueba temporal local
        test_filename = "prueba_conexion_notebook.txt"
        with open(test_filename, "w") as f:
            f.write("Archivo de prueba subido desde el Notebook de preparacion.")
            
        # Definir contenedor e intentar subir el archivo
        container_name = "raw"
        container_client = blob_service_client.get_container_client(container_name)
        blob_client = container_client.get_blob_client(test_filename)
        
        with open(test_filename, "rb") as data:
            blob_client.upload_blob(data, overwrite=True)
            
        print(f"[CONEXION EXITOSA] Se cargo el archivo '{test_filename}' en el contenedor '{container_name}'.")
        
        # Limpiar archivo de prueba local
        os.remove(test_filename)
        
    except Exception as e:
        print(f"[ERROR DE CONEXION] No se pudo cargar el archivo: {e}")


In [ ]:
# BLOQUE 3: BOCETO DEL DAG DE AIRFLOW (Guia de referencia para escribir en el repo)
# Este codigo sirve como plantilla para crear el archivo .py en la carpeta de dags de Airflow.

dag_template = """
from airflow import DAG
from airflow.providers.microsoft.azure.transfers.local_to_wasb import LocalFilesystemToWasbOperator
from airflow.operators.python import PythonOperator
from airflow.sensors.filesystem import FileSensor
from datetime import datetime, timedelta
import os
import shutil

# NOTA: Configura las rutas absolutas donde correra Airflow (usualmente dentro del contenedor docker)
INPUT_DIR = '/opt/airflow/dags/datos_simulados/input'
ARCHIVE_DIR = '/opt/airflow/dags/datos_simulados/archive'
CONTAINER_NAME = 'raw'

def archive_processed_files(**kwargs):
    # Mover archivos CSV procesados de input/ a archive/ para limpiar la bandeja
    files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.csv')]
    for file in files:
        src = os.path.join(INPUT_DIR, file)
        dst = os.path.join(ARCHIVE_DIR, file)
        shutil.move(src, dst)
        print(f'Archivo movido a historico: {file}')

default_args = {
    'owner': 'grupo-de-trabajo',
    'depends_on_past': False,
    'start_date': datetime(2026, 6, 1),
    'retries': 1,
    'retry_delay': timedelta(minutes=2),
}

with DAG(
    'de_proyecto_avance_dag',
    default_args=default_args,
    description='DAG para mover datos de servidor simulado local a Azure Data Lake',
    schedule_interval='@daily',
    catchup=False,
) as dag:

    # 1. Monitorear si hay archivos nuevos en la ruta local
    wait_for_file = FileSensor(
        task_id='esperar_archivos_locales',
        filepath='*.csv',
        fs_conn_id='fs_default', # Debe estar configurada en Airflow Connection apuntando a INPUT_DIR
        poke_interval=15,
        timeout=120
    )

    # 2. Cargar archivo local a Azure Blob Storage
    upload_to_azure = LocalFilesystemToWasbOperator(
        task_id='cargar_archivo_a_azure',
        file_path=os.path.join(INPUT_DIR, 'ventas_*.csv'), 
        container_name=CONTAINER_NAME,
        blob_name='ventas_ingesta/ventas_latest.csv',
        wasb_conn_id='wasb_default', # Configurado en Airflow Connection con el connection_string
        overwrite=True
    )

    # 3. Archivar archivos locales procesados
    archive_local = PythonOperator(
        task_id='archivar_archivos_locales',
        python_callable=archive_processed_files
    )

    wait_for_file >> upload_to_azure >> archive_local
"""

print("[INFO] Plantilla cargada. Copia este boceto a tu archivo .py en la carpeta de DAGs.")
